In [ ]:
# type: ignore


import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient


load_dotenv()

llm_model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME"),
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),
    extra_body={"enable_thinking": False},
)

embedding_model = DashScopeEmbeddings(
    model="text-embedding-v4",
    dashscope_api_key=os.environ.get("COMPATIBLE_API_KEY"),
)

client = QdrantClient(host="localhost", port=6333)

vectorstore = QdrantVectorStore(
    client=client,
    collection_name="agentic_rag_survey",
    embedding=embedding_model,
)

In [2]:
from langchain_core.documents import Document
from typing import List, TypedDict


class GraphState(TypedDict):
    """State object for workflow containing query, documents, and control flags."""

    question: str  # User's original query
    generation: str  # LLM-generated response
    web_search: bool  # Control flag for web search requirement
    documents: List[Document]  # Retrieved documents

In [18]:
# Workflow node identifiers
RETRIEVE = "retrieve"
GRADE_DOCUMENTS = "grade_documents"
WEBSEARCH = "web_search"
RAG_ANSWER = "rag_answer"
DIRECT_ANSWER = "direct_answer"

# Retrieval Node

In [4]:
from typing import Any, Dict


retriever = vectorstore.as_retriever()


def retrieve(state: GraphState) -> Dict[str, Any]:
    """Retrieve documents from vector store."""
    print("---RETRIEVE---")
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}

# Document Grading Node

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""

    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


structured_llm_document_grader = llm_model.with_structured_output(GradeDocuments)

system_prompt = """You are a grader assessing relevance of a retrieved document to a user question. \n 
    If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question.
    
    You MUST output a valid JSON object with ONLY the key "binary_score"."""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

retrieval_grader = grade_prompt | structured_llm_document_grader

In [6]:
def grade_documents(state: GraphState) -> Dict[str, Any]:
    """
    Determines whether the retrieved documents are relevant to the question
    If any document is not relevant, we will set a flag to run web search

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Filtered out irrelevant documents and updated web_search state
    """

    print("---CHECK DOCUMENT RELEVANCE TO QUESTION---")
    question = state["question"]
    documents = state["documents"]

    filtered_docs = []
    web_search = False
    for document in documents:
        score = retrieval_grader.invoke(
            {"question": question, "document": document.page_content}
        )
        grade = score.binary_score  # type: ignore
        if grade.lower() == "yes":
            print("---GRADE: DOCUMENT RELEVANT---")
            filtered_docs.append(document)
        else:
            print("---GRADE: DOCUMENT NOT RELEVANT---")
            web_search = True
            continue
    return {"documents": filtered_docs, "question": question, "web_search": web_search}

# Web Search Node

In [7]:
from langchain_tavily import TavilySearch


web_search_tool = TavilySearch(max_results=3)


def web_search(state: GraphState) -> Dict[str, Any]:
    print("---WEB SEARCH---")
    question = state["question"]

    # Initialize documents - this was the missing part!
    documents = state.get("documents", [])  # Get existing documents or empty list

    tavily_results = web_search_tool.invoke({"query": question})["results"]
    joined_tavily_result = "\n".join(
        [tavily_result["content"] for tavily_result in tavily_results]
    )
    web_results = Document(page_content=joined_tavily_result)

    # Add web results to existing documents (or create new list if documents was empty)
    if documents:
        documents.append(web_results)
    else:
        documents = [web_results]

    return {"documents": documents, "question": question}

# Answer Node

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


rag_prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:"""

rag_prompt = ChatPromptTemplate([("system", rag_prompt_template)])

rag_answer_chain = rag_prompt | llm_model | StrOutputParser()


def rag_answer(state: GraphState) -> Dict[str, Any]:
    """Generate answer using documents and question."""
    print("---RAG GENERATE---")
    question = state["question"]
    documents = state["documents"]
    generation = rag_answer_chain.invoke({"context": documents, "question": question})
    return {"documents": documents, "question": question, "generation": generation}

In [16]:
direct_answer_prompt_template = """
You are a helpful and concise AI assistant.  
Answer the user's question directly using your own knowledge.  
Use three sentences maximum and keep the answer concise.

Question: {question} 
"""

direct_answer_prompt = ChatPromptTemplate([("system", direct_answer_prompt_template)])

direct_answer_chain = direct_answer_prompt | llm_model


def direct_answer(state: GraphState) -> Dict[str, Any]:
    """Generate answer using LLM own knowledge."""
    print("---DIRECT GENERATE---")
    question = state["question"]
    generation = direct_answer_chain.invoke({"question": question})
    return {"question": question, "generation": generation}

# Router

In [19]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class RouteQuery(BaseModel):
    """Route a user query to the most relevant destination."""

    datasource: Literal["vector_store", "web_search", "direct_answer"] = Field(
        ...,
        description=(
            "Choose 'vector_store' for questions about Agentic RAG or RAG architectures; "
            "'web_search' for questions requiring up-to-date or external information; "
            "'direct_answer' for greetings, simple factual questions (e.g., capital of China), "
            "or anything the LLM can confidently answer without retrieval."
        ),
    )


structured_llm_router = llm_model.with_structured_output(RouteQuery)

system_prompt = system_prompt = """
You are an expert router that decides how to handle a user question.

- **Route to "vector_store"** if the question is about:
    • Agentic Retrieval-Augmented Generation (Agentic RAG)
    • Agent-based RAG systems
    • RAG workflow patterns (multi-agent, adaptive, self-correction, etc.)

- **Route to "web_search"** if the question:
    • Requires current information (e.g., stock prices, news, sports results)
    • Is about topics outside your training data or too niche
    • Cannot be answered confidently from general knowledge alone

- **Route to "direct_answer"** if the question is:
    • A greeting (e.g., "Hi", "Hello", "How are you?")
    • A simple factual question with a well-known answer (e.g., "What is the capital of China?", "Who wrote Hamlet?")
    • A basic conceptual question that doesn't need retrieval (e.g., "What is photosynthesis?")
    • Any query the LLM can answer directly and accurately from its internal knowledge

You MUST output a valid JSON object with ONLY the key "datasource"."""

route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

question_router = route_prompt | structured_llm_router

# Hallucination Grader

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in generation answer."""

    binary_score: bool = Field(
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )


structured_llm_hallucinations_grader = llm_model.with_structured_output(
    GradeHallucinations
)

system_prompt = """
You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. \n 
Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts.

You MUST output a valid JSON object with ONLY the key "binary_score"."""

hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "Set of facts: \n\n {documents} \n\n LLM generation: {generation}"),
    ]
)

hallucination_grader = hallucination_prompt | structured_llm_hallucinations_grader

# Answer Grader

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class GradeAnswer(BaseModel):
    """Binary score indicating whether a generated answer adequately addresses the user's question."""

    binary_score: bool = Field(
        description="Answer addresses the question, 'yes' or 'no'"
    )


structured_llm_answer_grader = llm_model.with_structured_output(GradeAnswer)

system_prompt = """You are a grader assessing whether an answer addresses / resolves a question \n 
     Give a binary score 'yes' or 'no'. Yes' means that the answer resolves the question.
     
     You MUST output a valid JSON object with ONLY the key "binary_score"."""

answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "User question: \n\n {question} \n\n LLM generation: {generation}"),
    ]
)

answer_grader = answer_prompt | structured_llm_answer_grader

# Complete Graph Workflow

In [25]:
# type: ignore


from langgraph.graph import END, StateGraph


def decide_to_generate(state: GraphState):
    """Determine whether to proceed with generation or search the web based on document relevance scores."""
    print("---ASSESS DOCUMENTS---")
    return WEBSEARCH if state["web_search"] else RAG_ANSWER


def grade_generation_grounded_in_documents_and_question(state):
    """Grade answer quality and groundedness."""
    print("---CHECK HALLUCINATIONS---")
    question = state["question"]
    documents = "\n\n".join([document.page_content for document in state["documents"]])
    generation = state["generation"]

    # Check if grounded
    score = hallucination_grader.invoke(
        {"documents": documents, "generation": generation}
    )

    if score.binary_score:
        # Check if useful
        score = answer_grader.invoke({"question": question, "generation": generation})
        return "useful" if score.binary_score else "not useful"
    else:
        return "not supported"


def route_question(state: GraphState) -> str:
    """Route question to vectorstore or web search or direct answer."""
    print("---ROUTE QUESTION---")
    source: RouteQuery = question_router.invoke({"question": state["question"]})

    if source.datasource == "vector_store":
        return RETRIEVE
    if source.datasource == "web_search":
        return WEBSEARCH
    if source.datasource == "direct_answer":
        return DIRECT_ANSWER


# Build workflow
workflow = StateGraph(GraphState)
workflow.add_node(RETRIEVE, retrieve)
workflow.add_node(GRADE_DOCUMENTS, grade_documents)
workflow.add_node(WEBSEARCH, web_search)
workflow.add_node(RAG_ANSWER, rag_answer)
workflow.add_node(DIRECT_ANSWER, direct_answer)

workflow.set_conditional_entry_point(
    route_question,
    {WEBSEARCH: WEBSEARCH, RETRIEVE: RETRIEVE, DIRECT_ANSWER: DIRECT_ANSWER},
)

workflow.add_edge(RETRIEVE, GRADE_DOCUMENTS)
workflow.add_conditional_edges(
    GRADE_DOCUMENTS,
    decide_to_generate,
    {WEBSEARCH: WEBSEARCH, RAG_ANSWER: RAG_ANSWER},
)
workflow.add_conditional_edges(
    RAG_ANSWER,
    grade_generation_grounded_in_documents_and_question,
    {"not supported": RAG_ANSWER, "useful": END, "not useful": WEBSEARCH},
)
workflow.add_edge(WEBSEARCH, RAG_ANSWER)
workflow.add_edge(DIRECT_ANSWER, END)

app = workflow.compile()

In [26]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	grade_documents(grade_documents)
	web_search(web_search)
	rag_answer(rag_answer)
	direct_answer(direct_answer)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> direct_answer;
	__start__ -.-> retrieve;
	__start__ -.-> web_search;
	grade_documents -.-> rag_answer;
	grade_documents -.-> web_search;
	rag_answer -. &nbsp;useful&nbsp; .-> __end__;
	rag_answer -. &nbsp;not useful&nbsp; .-> web_search;
	retrieve --> grade_documents;
	web_search --> rag_answer;
	direct_answer --> __end__;
	rag_answer -. &nbsp;not supported&nbsp; .-> rag_answer;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



# Test graph

In [27]:
app.invoke({"question": "hi"})  # type: ignore

---ROUTE QUESTION---
---DIRECT GENERATE---


{'question': 'hi',
 'generation': AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 53, 'total_tokens': 62, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3-32b', 'system_fingerprint': None, 'id': 'chatcmpl-129a1bd8-19d7-4919-92a9-6f6832597f2f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--11ef1eff-f5cd-4606-8890-6e4ce4017b74-0', usage_metadata={'input_tokens': 53, 'output_tokens': 9, 'total_tokens': 62, 'input_token_details': {}, 'output_token_details': {}})}

In [28]:
app.invoke({"question": "什么是 Agentic RAG"})  # type: ignore

---ROUTE QUESTION---
---RETRIEVE---
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---ASSESS DOCUMENTS---
---RAG GENERATE---
---CHECK HALLUCINATIONS---


{'question': '什么是 Agentic RAG',
 'generation': 'Agentic RAG 是一种结合了自主 AI 代理的 Retrieval-Augmented Generation（检索增强生成）系统，通过动态决策、迭代推理和协作工作流来提升复杂任务处理能力。它突破了传统 RAG 的静态流程限制，具备更高的灵活性和上下文感知能力。该系统适用于需要实时适应和精准响应的场景，如客户支持、金融分析和教育领域。',
 'web_search': False,
 'documents': [Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-02-05T01:26:00+00:00', 'author': '', 'keywords': '', 'moddate': '2025-02-05T01:26:00+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../example_data/A_SURVEY_ON_AGENTIC_RAG.pdf', 'total_pages': 39, 'page': 0, 'page_label': '1', 'start_index': 781, '_id': 'e984b0d2-381c-4368-81eb-9525475d40c6', '_collection_name': 'agentic_rag_survey'}, page_content='outputs. Retrieval-Augmented Generation (RAG) has emerged as a solution, enhancing LLMs by\nintegrating real-time data retrieval to provide contextually re

In [32]:
app.invoke({"question": "合肥的天气怎么样"})  # type: ignore

---ROUTE QUESTION---
---WEB SEARCH---
---RAG GENERATE---
---CHECK HALLUCINATIONS---


{'question': '合肥的天气怎么样',
 'generation': '合肥近期天气晴好，今天气温4~16°C，东南风2级，空气质量良。未来几天以晴转多云为主，温度适中。需要注意的是，周末可能会有小雨和降温。',
 'documents': [Document(metadata={}, page_content='合肥夏季熱、沈悶、潮濕和多雲時陰；冬季寒冷和多雲。 全年溫度一般在-1°C 至32°C 的範圍内，很少低於-5°C 或高於36°C。\n18°C10°C1°C 15天天气预报 * 昨天 1级 * 今天 1级 * 明天 3级 轻度 * 周二 1级 * 周三 1级 * 周四 3级 * 周五 3级 轻度 * 周六 3级 轻度 * 周日 1级 * 周一 1级 * 周二 1级 * 周三 1级 * 周四 1级 * 周五 1级 * 周六 1级 * 周日 1级 生活气象指数 下午好，今日天气晴好，心情也要保持晴朗哦 穿衣: 毛衣类 2℃~18℃ 出游: 较适宜 晴, 空气良 感冒: 易发 风力较大 洗车: 不适宜 近期有大风 未来40天天气预报 * 周日 * 周一 * 周二 * 周三 * 周四 * 周五 * 周六 * 今天 2 ~ 18°C 西北风1级 1 ~ 14°C 东风3级 轻度 3 ~ 15°C 东南风1级 5 ~ 18°C 西风1级 4 ~ 18°C 东北风3级 5 ~ 11°C 小雨 东风3-4级 轻度 3 ~ 9°C 多云 西北风3级 轻度 0 ~ 8°C 西风1级 1 ~ 14°C 东南风1级 4 ~ 14°C 西南风1级 3 ~ 14°C 东北风1级 7 ~ 15°C 西北风1级 2 ~ 13°C 小雨 西北风1级 3 ~ 9°C 小雨 西北风1级 -1 ~ 8°C 小雨 西北风1级 2 ~ 9°C 多云 1 ~ 9°C\n* 昨天 *12/08* *多云转晴* 2~15° 东北风2级 良 "昨天天气"). * 今天 *12/09* *晴转多云* 4~16° 东南风2级 良. * 明天 *12/10* *晴* 3~19° 西风1级 良. * 周四 *12/11* *晴转多云* 5~17° 东风2级 良. * 周六 *12/13* *晴* 1~8° 西北风3级 优. * 周日 *12/14* *晴* -3~8° 西南风2级 优. * 周一 *12